In [1]:
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

In [2]:
from pathlib import Path

conversion_dir = Path("results") / "mol_rep_conversion" / "v1.1" / "trainset" / "raw_responses"
ocr_dir = Path("results") / "mol_rep_ocr" / "v1.1" / "trainset" / "raw_responses"

import pandas as pd

all_result = pd.DataFrame()

for file in conversion_dir.glob("*.jsonl"):
    df = pd.read_json(file, lines=True)
    all_result = pd.concat([all_result, df], ignore_index=True)

In [3]:
from src.utils import compute_mol_metrics

data_df = all_result.copy()

data_df[['is_gt_valid', 'is_pred_valid', 'is_em', 'is_can_smiles_match', 'is_inchikey_match', 'tanimoto_sim', '_parse_status']] = data_df.apply(lambda row: compute_mol_metrics(row["completion"], row["raw_responses"], row["output_rep_type"]), axis=1, result_type="expand")

In [4]:
def aggregate_func(df):
    return pd.Series({
        "num_samples": df.shape[0],
        "gt_valid_ratio": df["is_gt_valid"].mean(),
        "pred_valid_ratio": df["is_pred_valid"].mean(),
        "em_ratio": df["is_em"].mean(),
        "can_smiles_match_ratio": df["is_can_smiles_match"].mean(),
        "inchikey_match_ratio": df["is_inchikey_match"].mean(),
        "tanimoto_sim_mean": df["tanimoto_sim"].mean(),
        "tanimoto_sim_std": df["tanimoto_sim"].std(),
    })

Conversion

In [5]:
data_df.groupby("model_name").apply(aggregate_func)

,num_samples,gt_valid_ratio,pred_valid_ratio,em_ratio,can_smiles_match_ratio,inchikey_match_ratio,tanimoto_sim_mean,tanimoto_sim_std
model_name,,,,,,,,
qwen3_vl_4b_i_sft_lora_ocr_conversion,6995.0,1.0,0.822588,0.644174,0.644889,0.644889,0.714222,0.42218


In [6]:
data_df[data_df._parse_status == "success"].groupby("model_name").apply(aggregate_func)

,num_samples,gt_valid_ratio,pred_valid_ratio,em_ratio,can_smiles_match_ratio,inchikey_match_ratio,tanimoto_sim_mean,tanimoto_sim_std
model_name,,,,,,,,
qwen3_vl_4b_i_sft_lora_ocr_conversion,5754.0,1.0,1.0,0.783107,0.783976,0.783976,0.868262,0.287947


In [7]:
(1 - 0.98213) * 6995

125.00065000000036

OCR

In [8]:
ocr_results = pd.DataFrame()

for file in ocr_dir.glob("*.jsonl"):
    df = pd.read_json(file, lines=True)
    ocr_results = pd.concat([ocr_results, df], ignore_index=True)

In [9]:
ocr_df = ocr_results.copy()

ocr_df[['is_gt_valid', 'is_pred_valid', 'is_em', 'is_can_smiles_match', 'is_inchikey_match', 'tanimoto_sim', '_parse_status']] = ocr_df.apply(lambda row: compute_mol_metrics(row["completion"], row["raw_responses"], row["output_rep_type"]), axis=1, result_type="expand")

In [10]:
ocr_df.groupby("model_name").apply(aggregate_func)

,num_samples,gt_valid_ratio,pred_valid_ratio,em_ratio,can_smiles_match_ratio,inchikey_match_ratio,tanimoto_sim_mean,tanimoto_sim_std
model_name,,,,,,,,
qwen3_vl_4b_i_sft_lora_ocr_conversion,640.0,1.0,0.653125,0.276562,0.279687,0.279687,0.408359,0.427076


In [11]:
ocr_df[ocr_df._parse_status == "success"].groupby("model_name").apply(aggregate_func)

,num_samples,gt_valid_ratio,pred_valid_ratio,em_ratio,can_smiles_match_ratio,inchikey_match_ratio,tanimoto_sim_mean,tanimoto_sim_std
model_name,,,,,,,,
qwen3_vl_4b_i_sft_lora_ocr_conversion,418.0,1.0,1.0,0.423445,0.42823,0.42823,0.625239,0.378904
